# 01 MPO・Hamiltonian environment・1-site局所有効問題

開境界の横磁場Ising鎖を使い、HamiltonianをMPOで表し、左右のHamiltonian environmentから1-site有効Hamiltonianを作ります。

Notebook 00では、全状態への写像 $P$ とfull Hamiltonianを明示的に作って1回の2-site局所更新を確認しました。今回は、同じ局所有効問題を**周囲の縮約結果から作る方法**へ進みます。

このNotebookの対象は1つの中心siteに対する1回の局所更新です。1-site sweep全体、2-site sweep、bond truncation、NNへの適用は扱いません。


## 1. 学習目的と穴埋め方針

次の依存関係を、1つの数値例で確認します。

$$
\text{HamiltonianのMPO}
\longrightarrow
\text{left / right environment}
\longrightarrow
H_{\mathrm{eff}}
\longrightarrow
\text{1-site局所更新}
$$

主な確認項目は次です。

1. MPOが、独立に作ったfull Hamiltonianと同じ演算子を表す。
2. 左右environmentが、bra・MPO・ketの固定部分を正しく縮約する。
3. environmentから作った$H_{\mathrm{eff}}$が、明示的な$P^\mathsf{T}HP$と一致する。
4. mixed-canonical formでは$P^\mathsf{T}P=I$となり、通常の固有値問題として局所最適化できる。
5. 1回の局所更新が最適化するのは、固定した周囲と現在のbond dimensionが定める部分空間である。

コード中の`___`が穴埋め箇所です。各TODOの直前には、実装する縮約の意味と満たすべき関係だけを示します。具体的なTensor shapeやAPI呼び出しはヒントとして示しません。

`___ = None`としているため、未回答の段階は`TODO`または`SKIP`となり、Notebook全体をRun Allできます。数値が得られた場合はassertを実行し、誤った実装をスキップしません。


In [10]:
from __future__ import annotations

import itertools
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
from pathlib import Path

import torch



def add_project_src_to_path(start: Path) -> Path:
    """プロジェクトのsrcをimport対象へ追加し、ルートを返す。"""
    for candidate in (start, *start.parents):
        src_dir = candidate / "src"
        if (src_dir / "nn_compression").is_dir():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return candidate
    raise FileNotFoundError("src/nn_compressionを含むプロジェクトルートが見つかりません。")


PROJECT_ROOT = add_project_src_to_path(Path.cwd())

from nn_compression.compression import (  # noqa: E402
    tt_canonicality_errors,
    tt_canonicalize,
    tt_fro_norm,
    tt_reconstruct,
)

DEVICE = torch.device("cpu")
DTYPE = torch.float64
ATOL = 1e-10
RTOL = 1e-10
SEED = 20261025

# 各TODOでは右辺の___をPyTorchコードへ置き換える。
___ = None

print("project root:", PROJECT_ROOT)
print("device / dtype:", DEVICE, DTYPE)


project root: d:\dev\nn-compression-svd-dmrg
device / dtype: cpu torch.float64


## 2. 数値例：開境界の横磁場Ising鎖

サイト数$L=6$の開境界系を使います。Pauli行列を$X,Z$とし、Hamiltonianを

$$
H
=
-J\sum_{i=1}^{L-1}Z_iZ_{i+1}
-g\sum_{i=1}^{L}X_i
$$

と定義します。今回は$J=1$、$g=1.2$です。$S^\alpha=\sigma^\alpha/2$ではなく、固有値が$\pm1$のPauli行列を使います。

まず、MPOとは独立にKronecker積からfull Hamiltonianを作ります。この行列はMPOと局所有効Hamiltonianを検証するための参照値であり、environmentによる更新処理には使いません。


In [11]:
LENGTH = 6
J_COUPLING = 1.0
TRANSVERSE_FIELD = 1.2

IDENTITY = torch.eye(2, dtype=DTYPE, device=DEVICE)
PAULI_X = torch.tensor(
    [[0.0, 1.0], [1.0, 0.0]], dtype=DTYPE, device=DEVICE
)
PAULI_Z = torch.tensor(
    [[1.0, 0.0], [0.0, -1.0]], dtype=DTYPE, device=DEVICE
)


def kron_all(operators: list[torch.Tensor]) -> torch.Tensor:
    """局所演算子をsite順にKronecker積する。"""
    result = operators[0]
    for operator in operators[1:]:
        result = torch.kron(result, operator)
    return result


def build_tfim_dense_hamiltonian(
    length: int,
    coupling: float,
    transverse_field: float,
) -> torch.Tensor:
    """開境界横磁場Ising鎖のfull Hamiltonianを独立に構築する。

    Args:
        length: サイト数。
        coupling: 隣接するZ演算子間の結合定数。
        transverse_field: 各siteへ作用するX項の係数。

    Returns:
        計算基底で表したfull Hamiltonian。
    """
    hilbert_dimension = 2**length
    hamiltonian = torch.zeros(
        (hilbert_dimension, hilbert_dimension), dtype=DTYPE, device=DEVICE
    )

    for site in range(length):
        operators = [IDENTITY] * length
        operators = operators.copy()
        operators[site] = PAULI_X
        hamiltonian = hamiltonian - transverse_field * kron_all(operators)

    for site in range(length - 1):
        operators = [IDENTITY] * length
        operators = operators.copy()
        operators[site] = PAULI_Z
        operators[site + 1] = PAULI_Z
        hamiltonian = hamiltonian - coupling * kron_all(operators)

    return hamiltonian


H_DENSE = build_tfim_dense_hamiltonian(
    LENGTH,
    J_COUPLING,
    TRANSVERSE_FIELD,
)

torch.testing.assert_close(H_DENSE, H_DENSE.T, atol=ATOL, rtol=RTOL)
print("dense Hamiltonian reference: OK")


dense Hamiltonian reference: OK


## 3. 第1段階：HamiltonianをMPOで表す

### 目的

MPO coreの添字順序を

$$
W^{[k]}_{a_{k-1},\,s'_k,\,s_k,\,a_k}
$$

とします。$s_k$は入力側、$s'_k$は出力側の物理添字です。

今回のHamiltonianは、次の演算子値行列で表せます。

$$
W^{[1]}
=
\begin{pmatrix}
-gX & -JZ & I
\end{pmatrix},
$$

$$
W^{[k]}
=
\begin{pmatrix}
I & 0 & 0\\
Z & 0 & 0\\
-gX & -JZ & I
\end{pmatrix}
\qquad (1<k<L),
$$

$$
W^{[L]}
=
\begin{pmatrix}
I\\
Z\\
-gX
\end{pmatrix}.
$$

各要素は数ではなく局所演算子です。境界coreとbulk coreを組み立て、MPOを全縮約した結果が`H_DENSE`と一致するようにしてください。

### 期待する検証結果

- MPOが表す演算子は実対称である。
- MPOをdense化した演算子と、独立に構築した`H_DENSE`が許容誤差内で一致する。


In [12]:
def build_tfim_mpo(
    length: int,
    coupling: float,
    transverse_field: float,
) -> list[torch.Tensor] | None:
    """開境界横磁場Ising鎖のHamiltonian MPOを作る。

    Args:
        length: サイト数。
        coupling: 隣接するZ演算子間の結合定数。
        transverse_field: 各siteへ作用するX項の係数。

    Returns:
        左境界から右境界へ並べたMPO core。未実装ならNone。
    """
    if length < 2:
        raise ValueError("この演習ではlength >= 2を対象とします。")

    zero = torch.zeros_like(IDENTITY)

    x_term = -transverse_field * PAULI_X
    z_start = -coupling * PAULI_Z

    # (-gX, -JZ, I)
    # (出力, 入力, 右の結合) → 左の結合の次元1を追加
    left_boundary = torch.stack(
        [x_term, z_start, IDENTITY], dim=-1
    ).unsqueeze(0)
    if left_boundary is None:
        return None

    # [[ I,   0, 0],
    #  [ Z,   0, 0],
    #  [-gX, -JZ, I]]
    bulk = torch.stack(
        [
            torch.stack([IDENTITY, zero, zero], dim=-1),
            torch.stack([PAULI_Z, zero, zero], dim=-1),
            torch.stack([x_term, z_start, IDENTITY], dim=-1),
        ],
        dim=0,
    )
    if bulk is None:
        return None

    # (I, Z, -gX)^T
    # (左の結合, 出力, 入力) → 右の結合の次元1を追加
    right_boundary = torch.stack(
        [IDENTITY, PAULI_Z, x_term], dim=0
    ).unsqueeze(-1)
    if right_boundary is None:
        return None

    return [left_boundary] + [bulk.clone() for _ in range(length - 2)] + [right_boundary]


def mpo_to_dense(mpo_cores: list[torch.Tensor]) -> torch.Tensor:
    """検証用に、小さいMPOを計算基底のdense行列へ戻す。

    Args:
        mpo_cores: 左から右へ並べたMPO core。

    Returns:
        MPOが表すdense演算子。
    """
    physical_dimension = mpo_cores[0].shape[1]
    configurations = list(
        itertools.product(range(physical_dimension), repeat=len(mpo_cores))
    )
    dimension = physical_dimension ** len(mpo_cores)
    dense = torch.zeros((dimension, dimension), dtype=DTYPE, device=DEVICE)

    for row, output_configuration in enumerate(configurations):
        for column, input_configuration in enumerate(configurations):
            bond_vector = torch.ones(1, dtype=DTYPE, device=DEVICE)
            for core, output_index, input_index in zip(
                mpo_cores,
                output_configuration,
                input_configuration,
            ):
                bond_vector = bond_vector @ core[:, output_index, input_index, :]
            dense[row, column] = bond_vector.reshape(())

    return dense


MPO_CORES = build_tfim_mpo(LENGTH, J_COUPLING, TRANSVERSE_FIELD)

if MPO_CORES is not None:
    H_FROM_MPO = mpo_to_dense(MPO_CORES)
    torch.testing.assert_close(H_FROM_MPO, H_DENSE, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(H_FROM_MPO, H_FROM_MPO.T, atol=ATOL, rtol=RTOL)
    print("stage 1: OK")
else:
    H_FROM_MPO = None
    print("stage 1: TODO")


stage 1: OK


## 4. mixed-canonical MPSを用意する

局所有効問題を通常の固有値問題として解くため、中心より左を左正準形、右を右正準形にします。これにより、中心から全状態への写像$P$は

$$
P^\mathsf{T}P=I
$$

を満たします。

ここでは、既存APIの`tt_canonicalize`、`tt_canonicality_errors`、`tt_fro_norm`、`tt_reconstruct`を再利用します。正準化処理をNotebook内で作り直しません。

初期MPSのbond dimensionは2に固定します。したがって、1-site更新が探索できるのは全Hilbert空間ではなく、固定した左右ブロックが張る部分空間です。


In [13]:
CENTER_SITE = 2
INITIAL_BOND_DIMENSION = 2

generator = torch.Generator(device="cpu").manual_seed(SEED)
ranks = [1] + [INITIAL_BOND_DIMENSION] * (LENGTH - 1) + [1]

raw_mps_cores = [
    torch.randn(
        ranks[site],
        2,
        ranks[site + 1],
        generator=generator,
        dtype=DTYPE,
        device=DEVICE,
    )
    for site in range(LENGTH)
]

MPS_CORES = tt_canonicalize(raw_mps_cores, center=CENTER_SITE)
mps_norm = tt_fro_norm(MPS_CORES)
MPS_CORES[CENTER_SITE] = MPS_CORES[CENTER_SITE] / mps_norm

canonicality = tt_canonicality_errors(MPS_CORES, center=CENTER_SITE)
canonicality_errors = canonicality["left"] + canonicality["right"]
assert canonicality_errors
assert max(error.item() for error in canonicality_errors) <= ATOL

INITIAL_STATE = tt_reconstruct(MPS_CORES).reshape(-1)
torch.testing.assert_close(
    torch.linalg.vector_norm(INITIAL_STATE),
    torch.tensor(1.0, dtype=DTYPE, device=DEVICE),
    atol=ATOL,
    rtol=RTOL,
)

print("mixed-canonical MPS: OK")
print("center site:", CENTER_SITE)
print("initial bond dimension:", INITIAL_BOND_DIMENSION)


mixed-canonical MPS: OK
center site: 2
initial bond dimension: 2


## 5. 第2段階：left environmentを更新する

### 目的

left environmentは、更新対象より左にあるbra MPS・MPO・ket MPSを縮約した中間量です。

現在のenvironmentを$L^{[k]}$とすると、次のsiteを取り込む更新は

$$
L^{[k+1]}_{\alpha'_k,a_k,\alpha_k}
=
\sum_{
\alpha'_{k-1},a_{k-1},\alpha_{k-1},s'_k,s_k
}
L^{[k]}_{\alpha'_{k-1},a_{k-1},\alpha_{k-1}}
A^{[k]}_{\alpha'_{k-1},s'_k,\alpha'_k}
W^{[k]}_{a_{k-1},s'_k,s_k,a_k}
A^{[k]}_{\alpha_{k-1},s_k,\alpha_k}
$$

です。今回は実数なのでbra側の複素共役は現れません。

### 期待する検証結果

全siteを取り込んだ最後のenvironmentはスカラーになり、MPOを使った期待値とfull stateから計算した期待値が一致します。


In [14]:
def update_left_environment(
    environment: torch.Tensor,
    mps_core: torch.Tensor,
    mpo_core: torch.Tensor,
) -> torch.Tensor | None:
    """左側からbra MPS・MPO・ket MPSを1 site分取り込む。

    Args:
        environment: 現在までの左側縮約。
        mps_core: 次に取り込むMPS core。
        mpo_core: 同じsiteのMPO core。

    Returns:
        1 site分更新したleft environment。未実装ならNone。
    """
    # TODO: 数式に従い、共有する5つの添字を縮約する。
    updated_environment = torch.einsum(
        "abc,adi,bdej,cek->ijk",
        environment,
        mps_core.conj(),
        mpo_core,
        mps_core,
    )
    if updated_environment is None:
        return None
    return updated_environment


def build_left_environments(
    mps_cores: list[torch.Tensor],
    mpo_cores: list[torch.Tensor],
) -> list[torch.Tensor] | None:
    """各cut直前までを含むleft environmentを順番に作る。"""
    environment = torch.ones((1, 1, 1), dtype=DTYPE, device=DEVICE)
    environments = [environment]

    for mps_core, mpo_core in zip(mps_cores, mpo_cores):
        environment = update_left_environment(environment, mps_core, mpo_core)
        if environment is None:
            return None
        environments.append(environment)

    return environments


LEFT_ENVIRONMENTS = (
    build_left_environments(MPS_CORES, MPO_CORES)
    if MPO_CORES is not None
    else None
)

if LEFT_ENVIRONMENTS is not None:
    LEFT_EXPECTATION = LEFT_ENVIRONMENTS[-1].reshape(())
    DENSE_EXPECTATION = INITIAL_STATE @ H_DENSE @ INITIAL_STATE
    torch.testing.assert_close(
        LEFT_EXPECTATION,
        DENSE_EXPECTATION,
        atol=ATOL,
        rtol=RTOL,
    )
    print("stage 2: OK")
else:
    LEFT_EXPECTATION = DENSE_EXPECTATION = None
    print("stage 2: SKIP (MPO or left-environment update is TODO)")


stage 2: OK


## 6. 第3段階：right environmentを更新する

### 目的

right environmentは、更新対象より右にあるbra MPS・MPO・ket MPSを縮約した中間量です。右端から左へsiteを取り込みます。

現在のenvironmentを$R^{[k+1]}$とすると、更新は

$$
R^{[k]}_{\alpha'_{k-1},a_{k-1},\alpha_{k-1}}
=
\sum_{
\alpha'_k,a_k,\alpha_k,s'_k,s_k
}
A^{[k]}_{\alpha'_{k-1},s'_k,\alpha'_k}
W^{[k]}_{a_{k-1},s'_k,s_k,a_k}
A^{[k]}_{\alpha_{k-1},s_k,\alpha_k}
R^{[k+1]}_{\alpha'_k,a_k,\alpha_k}
$$

です。

### 期待する検証結果

右端から全siteを取り込んだスカラーも、left environmentおよびfull stateから得た期待値と一致します。


In [15]:
def update_right_environment(
    environment: torch.Tensor,
    mps_core: torch.Tensor,
    mpo_core: torch.Tensor,
) -> torch.Tensor | None:
    """右側からbra MPS・MPO・ket MPSを1 site分取り込む。

    Args:
        environment: 現在までの右側縮約。
        mps_core: 次に取り込むMPS core。
        mpo_core: 同じsiteのMPO core。

    Returns:
        1 site分更新したright environment。未実装ならNone。
    """
    # TODO: 数式に従い、共有する5つの添字を縮約する。
    updated_environment = torch.einsum(
        "ijk,adi,bdej,cek->abc",
        environment,
        mps_core.conj(),
        mpo_core,
        mps_core,
    )
    if updated_environment is None:
        return None
    return updated_environment


def build_right_environments(
    mps_cores: list[torch.Tensor],
    mpo_cores: list[torch.Tensor],
) -> list[torch.Tensor] | None:
    """各cut直後から右端までを含むright environmentを順番に作る。"""
    environment = torch.ones((1, 1, 1), dtype=DTYPE, device=DEVICE)
    environments: list[torch.Tensor | None] = [None] * (len(mps_cores) + 1)
    environments[-1] = environment

    for site in range(len(mps_cores) - 1, -1, -1):
        environment = update_right_environment(
            environment,
            mps_cores[site],
            mpo_cores[site],
        )
        if environment is None:
            return None
        environments[site] = environment

    assert all(value is not None for value in environments)
    return [value for value in environments if value is not None]


RIGHT_ENVIRONMENTS = (
    build_right_environments(MPS_CORES, MPO_CORES)
    if MPO_CORES is not None
    else None
)

if RIGHT_ENVIRONMENTS is not None:
    RIGHT_EXPECTATION = RIGHT_ENVIRONMENTS[0].reshape(())
    DENSE_EXPECTATION = INITIAL_STATE @ H_DENSE @ INITIAL_STATE
    torch.testing.assert_close(
        RIGHT_EXPECTATION,
        DENSE_EXPECTATION,
        atol=ATOL,
        rtol=RTOL,
    )
    if LEFT_EXPECTATION is not None:
        torch.testing.assert_close(
            RIGHT_EXPECTATION,
            LEFT_EXPECTATION,
            atol=ATOL,
            rtol=RTOL,
        )
    print("stage 3: OK")
else:
    RIGHT_EXPECTATION = None
    print("stage 3: SKIP (MPO or right-environment update is TODO)")


stage 3: OK


## 7. 第4段階：environmentから$H_{\mathrm{eff}}$を作る

### 目的

中心siteより左を$L$、右を$R$へ縮約済みとします。中心coreのbra側の要素とket側の要素の間に作用する局所有効Hamiltonianは、

$$
(H_{\mathrm{eff}})_{
(\alpha',s',\beta'),
(\alpha,s,\beta)
}
=
\sum_{a,b}
L_{\alpha',a,\alpha}
W_{a,s',s,b}
R_{\beta',b,\beta}
$$

です。

TODOでは、まず6添字のTensorとして縮約し、その後でbra側とket側をそれぞれ1本の添字へまとめます。

### 期待する検証結果

- $H_{\mathrm{eff}}$は実対称である。
- 現在の中心coreから計算した局所Rayleigh商が、full stateのエネルギーと一致する。


In [16]:
def build_one_site_effective_hamiltonian(
    left_environment: torch.Tensor,
    center_mpo_core: torch.Tensor,
    right_environment: torch.Tensor,
    center_core: torch.Tensor,
) -> torch.Tensor | None:
    """左右environmentと中心MPO coreから1-site有効Hamiltonianを作る。

    Args:
        left_environment: 中心より左をすべて含む縮約。
        center_mpo_core: 中心siteのMPO core。
        right_environment: 中心より右をすべて含む縮約。
        center_core: 局所変数の添字順を与える現在の中心core。

    Returns:
        中心coreのベクトルへ作用する有効Hamiltonian。未実装ならNone。
    """
    # TODO: L・中心W・Rを縮約し、bra側の添字を先、ket側を後に並べる。
    effective_tensor = torch.einsum(
        "abc,bdej,ijk->adicek",
        left_environment,   # L
        center_mpo_core,    # W：中心のHamiltonianの部品。Aではない
        right_environment, # R
    )
    if effective_tensor is None:
        return None

    local_dimension = center_core.numel()
    return effective_tensor.reshape(local_dimension, local_dimension)


if (
    MPO_CORES is not None
    and LEFT_ENVIRONMENTS is not None
    and RIGHT_ENVIRONMENTS is not None
):
    H_EFFECTIVE_ENVIRONMENT = build_one_site_effective_hamiltonian(
        LEFT_ENVIRONMENTS[CENTER_SITE],
        MPO_CORES[CENTER_SITE],
        RIGHT_ENVIRONMENTS[CENTER_SITE + 1],
        MPS_CORES[CENTER_SITE],
    )
else:
    H_EFFECTIVE_ENVIRONMENT = None

if H_EFFECTIVE_ENVIRONMENT is not None:
    center_vector = MPS_CORES[CENTER_SITE].reshape(-1)
    local_energy = (
        center_vector @ H_EFFECTIVE_ENVIRONMENT @ center_vector
    ) / (center_vector @ center_vector)
    dense_energy = INITIAL_STATE @ H_DENSE @ INITIAL_STATE

    torch.testing.assert_close(
        H_EFFECTIVE_ENVIRONMENT,
        H_EFFECTIVE_ENVIRONMENT.T,
        atol=ATOL,
        rtol=RTOL,
    )
    torch.testing.assert_close(local_energy, dense_energy, atol=ATOL, rtol=RTOL)
    print("stage 4: OK")
else:
    center_vector = local_energy = dense_energy = None
    print("stage 4: SKIP (environment or effective-Hamiltonian contraction is TODO)")


stage 4: OK


## 8. 明示的な$P^\mathsf{T}HP$との照合

この節は新しい実装課題ではなく、Notebook 00で学んだ写像$P$を使う検証です。

中心空間の基底を1本ずつMPSの中心へ入れ、full stateを列として並べます。mixed-canonical formであるため、

$$
P^\mathsf{T}P=I
$$

となります。そのうえで、

$$
H_{\mathrm{eff}}^{(\mathrm{env})}
=
P^\mathsf{T}H_{\mathrm{dense}}P
$$

を確認します。

実際のDMRG更新では$P$もfull Hamiltonianも作りません。ここではenvironmentの縮約が正しいことを確かめるためだけに使います。


In [17]:
def build_one_site_projection(
    mps_cores: list[torch.Tensor],
    center_site: int,
) -> torch.Tensor:
    """固定した周囲から、中心coreを全状態へ写す行列を作る。

    Args:
        mps_cores: mixed-canonical formのMPS core列。
        center_site: 変数として扱う中心site。

    Returns:
        中心空間の各基底を全状態へ写した列を持つ行列。
    """
    center_core = mps_cores[center_site]
    basis = torch.eye(
        center_core.numel(),
        dtype=center_core.dtype,
        device=center_core.device,
    )
    columns: list[torch.Tensor] = []

    for basis_vector in basis:
        candidate_cores = [core.clone() for core in mps_cores]
        candidate_cores[center_site] = basis_vector.reshape_as(center_core)
        columns.append(tt_reconstruct(candidate_cores).reshape(-1))

    return torch.stack(columns, dim=1)


PROJECTION = build_one_site_projection(MPS_CORES, CENTER_SITE)
projection_identity = torch.eye(
    PROJECTION.shape[1], dtype=DTYPE, device=DEVICE
)
torch.testing.assert_close(
    PROJECTION.T @ PROJECTION,
    projection_identity,
    atol=ATOL,
    rtol=RTOL,
)

H_EFFECTIVE_PROJECTION = PROJECTION.T @ H_DENSE @ PROJECTION

if H_EFFECTIVE_ENVIRONMENT is not None:
    torch.testing.assert_close(
        H_EFFECTIVE_ENVIRONMENT,
        H_EFFECTIVE_PROJECTION,
        atol=ATOL,
        rtol=RTOL,
    )
    print("projection and environment: OK")
else:
    print("projection: OK / environment comparison: SKIP")


projection and environment: OK


## 9. 補助演習：$H_{\mathrm{eff}}$を作らずに、その作用を計算する

### 学習目的

固定されたleft environment $L$、中心MPO core $W$、right environment $R$と、任意の入力Tensor $X$から、出力Tensor $Y$を直接計算します。

- $X$は中央MPS coreと同じshapeを持つ入力配列です。
- $X$は新しい物理的対象ではなく、局所空間へ入れる任意の候補です。
- $Y$は$X$と同じshapeを持つ出力配列です。
- $x,y$は、それぞれ$X,Y$を既存と同じ順序でflattenしたベクトルです。
- ここでのmatrix-freeは、$H_{\mathrm{eff}}$の密行列や6添字のeffective tensorを明示的に作らないという意味です。

既存の添字規約を使うと、作用は

$$
Y_{\alpha',s',\beta'}
=
\sum_{\alpha,s,\beta,a,b}
L_{\alpha',a,\alpha}
W_{a,s',s,b}
R_{\beta',b,\beta}
X_{\alpha,s,\beta}
$$

です。flattenすると、

$$
y=H_{\mathrm{eff}}x
$$

に対応します。

入力側の$X$を下から接続し、出力側の3添字だけを残すと考えます。

```text
      α'          s'          β'
       │           │           │
       L ─── a ─── W ─── b ─── R
       │           │           │
       α           s           β
        \__________ X __________/
```

TODOでは$L,W,R,X$を一度に直接縮約してください。`build_one_site_effective_hamiltonian`を呼んだり、$H_{\mathrm{eff}}$の密行列や6添字Tensorを途中で作ったりしてはいけません。添字の対応は上式から読み取り、完成した縮約式を記述してください。

### 検証する量

1. matrix-free作用と、明示的な$H_{\mathrm{eff}}$による行列ベクトル積。
2. 二次形式$x^\mathsf{T}y$と$x^\mathsf{T}H_{\mathrm{eff}}x$。どちらも1つのスカラーです。
3. 中央coreだけを$X$へ置き換えた全状態$\psi(X)$に対する期待値の分子。
4. mixed-canonical formから得られる分母$\psi(X)^\mathsf{T}\psi(X)=x^\mathsf{T}x$。
5. 分子と分母から作る全状態側・局所空間側のRayleigh商。

$X$と$\psi(X)$は規格化しないため、分子だけをエネルギーとは呼びません。


In [18]:
def apply_one_site_effective_hamiltonian(
    left_environment: torch.Tensor,
    center_mpo_core: torch.Tensor,
    right_environment: torch.Tensor,
    input_core: torch.Tensor,
) -> torch.Tensor | None:
    """1-site有効Hamiltonianの作用を密行列化せずに計算する。

    Args:
        left_environment: 中心より左をすべて含む縮約。
        center_mpo_core: 中心siteのMPO core。
        right_environment: 中心より右をすべて含む縮約。
        input_core: 局所空間へ入力する任意の候補Tensor。

    Returns:
        input_coreと同じshapeを持つ出力Tensor。未実装ならNone。
    """
    # TODO: L・W・R・Xを直接縮約し、出力側の3添字だけを残す。
    result = torch.einsum(
        "abc,bdej,ijk,cek->adi",
        left_environment,
        center_mpo_core,
        right_environment,
        input_core,
    )
    if result is None:
        return None
    return result


# 既存のMPS初期化に使った乱数列へ影響しない、補助演習専用generator。
matrix_free_generator = torch.Generator(device="cpu").manual_seed(SEED + 1)
MATRIX_FREE_INPUT = torch.randn(
    MPS_CORES[CENTER_SITE].shape,
    generator=matrix_free_generator,
    dtype=DTYPE,
    device=DEVICE,
)

matrix_free_dependencies_ready = (
    MPO_CORES is not None
    and LEFT_ENVIRONMENTS is not None
    and RIGHT_ENVIRONMENTS is not None
    and H_EFFECTIVE_ENVIRONMENT is not None
)

if matrix_free_dependencies_ready:
    MATRIX_FREE_OUTPUT = apply_one_site_effective_hamiltonian(
        LEFT_ENVIRONMENTS[CENTER_SITE],
        MPO_CORES[CENTER_SITE],
        RIGHT_ENVIRONMENTS[CENTER_SITE + 1],
        MATRIX_FREE_INPUT,
    )
else:
    MATRIX_FREE_OUTPUT = None

if MATRIX_FREE_OUTPUT is not None:
    matrix_free_input_vector = MATRIX_FREE_INPUT.reshape(-1)
    matrix_free_output_vector = MATRIX_FREE_OUTPUT.reshape(-1)
    explicit_output_vector = (
        H_EFFECTIVE_ENVIRONMENT @ matrix_free_input_vector
    )

    # 1. matrix-free作用と明示行列による作用を比較する。
    torch.testing.assert_close(
        matrix_free_output_vector,
        explicit_output_vector,
        atol=ATOL,
        rtol=RTOL,
    )

    # 2. 二次形式は、局所空間で計算した1つのスカラーである。
    matrix_free_quadratic_form = (
        matrix_free_input_vector @ matrix_free_output_vector
    )
    explicit_quadratic_form = (
        matrix_free_input_vector
        @ H_EFFECTIVE_ENVIRONMENT
        @ matrix_free_input_vector
    )
    assert matrix_free_quadratic_form.ndim == 0
    assert explicit_quadratic_form.ndim == 0
    torch.testing.assert_close(
        matrix_free_quadratic_form,
        explicit_quadratic_form,
        atol=ATOL,
        rtol=RTOL,
    )

    # 3. 中央coreだけをXへ置き換え、全状態側の期待値の分子と比較する。
    candidate_mps_cores = [core.clone() for core in MPS_CORES]
    candidate_mps_cores[CENTER_SITE] = MATRIX_FREE_INPUT.clone()
    candidate_state = tt_reconstruct(candidate_mps_cores).reshape(-1)
    full_state_numerator = candidate_state @ H_DENSE @ candidate_state
    torch.testing.assert_close(
        full_state_numerator,
        matrix_free_quadratic_form,
        atol=ATOL,
        rtol=RTOL,
    )

    # 4. mixed-canonicalな周囲により、全状態側と局所空間側の分母が一致する。
    full_state_denominator = candidate_state @ candidate_state
    local_denominator = matrix_free_input_vector @ matrix_free_input_vector
    torch.testing.assert_close(
        full_state_denominator,
        local_denominator,
        atol=ATOL,
        rtol=RTOL,
    )

    # 5. 分子と分母の一致から、正規化したRayleigh商も一致する。
    full_state_rayleigh_quotient = (
        full_state_numerator / full_state_denominator
    )
    local_rayleigh_quotient = matrix_free_quadratic_form / local_denominator
    torch.testing.assert_close(
        full_state_rayleigh_quotient,
        local_rayleigh_quotient,
        atol=ATOL,
        rtol=RTOL,
    )

    print("matrix-free effective-Hamiltonian action: OK")
else:
    matrix_free_input_vector = matrix_free_output_vector = None
    explicit_output_vector = None
    matrix_free_quadratic_form = explicit_quadratic_form = None
    candidate_mps_cores = candidate_state = None
    full_state_numerator = full_state_denominator = None
    local_denominator = None
    full_state_rayleigh_quotient = local_rayleigh_quotient = None
    if matrix_free_dependencies_ready:
        print("matrix-free effective-Hamiltonian action: TODO")
    else:
        print("matrix-free effective-Hamiltonian action: SKIP (a dependency is TODO)")


matrix-free effective-Hamiltonian action: OK


matrix-freeな作用を関数として渡せれば、将来は密な$H_{\mathrm{eff}}$を作らず反復固有値solverへ接続できます。今回は作用の一致確認までとし、solverの導出や実装には進みません。


## 10. 1回の1-site局所更新

`lowest_eigenpair`はNotebook 00で実装済みのため、ここでは完成コードとして再利用します。

$$
H_{\mathrm{eff}}c_{\mathrm{opt}}
=
E_{\mathrm{opt}}c_{\mathrm{opt}}
$$

の最小固有ベクトルを中心coreへ戻します。左右のcoreは固定したままです。

### 区別する値

- `energy_before`：更新前のMPSのエネルギー。
- `local_minimum_energy`：固定した周囲の局所有効問題で得た最小値。
- `energy_after`：更新後MPSをfull stateへ戻して再計算したエネルギー。
- `exact_ground_energy`：全Hilbert空間で厳密対角化した基底エネルギー。
- `global_residual`：更新後状態が全Hamiltonianの固有状態にどの程度近いか。

`energy_after`と`local_minimum_energy`は一致します。一方、1回の局所更新で`exact_ground_energy`へ到達することは要求しません。


In [19]:
def lowest_eigenpair(
    symmetric_operator: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """実対称行列の最小固有対を返す。"""
    eigenvalues, eigenvectors = torch.linalg.eigh(symmetric_operator)
    lowest_index = torch.argmin(eigenvalues)
    return eigenvalues[lowest_index], eigenvectors[:, lowest_index]


def normalized_energy(state: torch.Tensor, hamiltonian: torch.Tensor) -> torch.Tensor:
    """状態のRayleigh商を返す。"""
    return (state @ hamiltonian @ state) / (state @ state)


if H_EFFECTIVE_ENVIRONMENT is not None:
    local_minimum_energy, optimized_center_vector = lowest_eigenpair(
        H_EFFECTIVE_ENVIRONMENT
    )
    UPDATED_MPS_CORES = [core.clone() for core in MPS_CORES]
    UPDATED_MPS_CORES[CENTER_SITE] = optimized_center_vector.reshape_as(
        MPS_CORES[CENTER_SITE]
    )

    UPDATED_STATE = tt_reconstruct(UPDATED_MPS_CORES).reshape(-1)
    energy_before = normalized_energy(INITIAL_STATE, H_DENSE)
    energy_after = normalized_energy(UPDATED_STATE, H_DENSE)
    exact_ground_energy = torch.linalg.eigvalsh(H_DENSE)[0]
    local_residual = torch.linalg.vector_norm(
        H_EFFECTIVE_ENVIRONMENT @ optimized_center_vector
        - local_minimum_energy * optimized_center_vector
    )
    global_residual = torch.linalg.vector_norm(
        H_DENSE @ UPDATED_STATE - energy_after * UPDATED_STATE
    ) / torch.linalg.vector_norm(UPDATED_STATE)

    torch.testing.assert_close(
        energy_after,
        local_minimum_energy,
        atol=ATOL,
        rtol=RTOL,
    )
    assert energy_after.item() <= energy_before.item() + ATOL
    assert local_residual.item() <= ATOL

    print("stage 5: OK")
    print("energy before:", energy_before.item())
    print("local minimum / energy after:", energy_after.item())
    print("exact ground energy (reference):", exact_ground_energy.item())
    print("global residual (reference):", global_residual.item())
else:
    UPDATED_MPS_CORES = UPDATED_STATE = None
    energy_before = local_minimum_energy = energy_after = None
    exact_ground_energy = local_residual = global_residual = None
    print("stage 5: SKIP (effective Hamiltonian is TODO)")


stage 5: OK
energy before: 1.157791565571693
local minimum / energy after: -2.805281614274409
exact ground energy (reference): -8.269344348226955
global residual (reference): 3.2695724304911513


## 11. 最後の整理問題

コードの結果を根拠に、次を短く説明してください。完成文は用意していません。

1. full HamiltonianとMPOは、同じ演算子をどのように異なる形式で保持しているか。
2. left environmentには、更新対象のどちら側のsiteが含まれているか。right environmentについても説明してください。
3. inner product用のenvironmentと、今回のHamiltonian environmentでは、縮約に含まれるTensorがどう異なるか。
4. mixed-canonical formで$P^\mathsf{T}P=I$になることが、通常の固有値問題とどう関係するか。
5. environmentから作った$H_{\mathrm{eff}}$と$P^\mathsf{T}HP$が一致することは、何を検証しているか。
6. `energy_after`が`local_minimum_energy`と一致しても、`exact_ground_energy`と一致するとは限らない理由。
7. 今回の1-site更新ではbond dimensionを増やせない理由。
8. このNotebookを完了しても、1-siteまたは2-siteのsweep全体を実装したことにならない理由。

## 完了条件

- MPOと独立なfull Hamiltonianが一致する。
- 左右どちらから全縮約しても同じ期待値になる。
- environment由来と明示的射影由来の$H_{\mathrm{eff}}$が一致する。
- 正準条件と$P^\mathsf{T}P=I$が確認できる。
- 局所更新後の全状態エネルギーが局所最小固有値と一致する。
- EDとの差とglobal residualは診断値として表示し、1回の局所更新の合格条件にはしない。

次のNotebookでは、同じHamiltonianを使って2-site局所更新を複数bondへつなぎ、left-to-right / right-to-leftの往復sweep、environmentの逐次更新、SVD打ち切り、discarded weight、収束判定を扱います。
